In [1]:
%pip install xgboost
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import xgboost as xgb

# 1. โหลดข้อมูล
train_df = pd.read_csv("../cleaned_data/train_data.csv")
test_df = pd.read_csv("../cleaned_data/test_data.csv")

# 2. แยก Features (X) และ Target (y)
X_train = train_df.drop(columns=['y'])
y_train = train_df['y']
X_test = test_df.drop(columns=['y'])
y_test = test_df['y']

# 3. ทำ One-Hot Encoding และจัดคอลัมน์ให้ตรงกัน
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# 4. คำนวณสัดส่วนสำหรับถ่วงน้ำหนัก (จำนวนคนที่ตอบ No หารด้วยคนที่ตอบ Yes)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# 5. สร้างและเทรนโมเดล XGBoost
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight, 
    random_state=0, 
    eval_metric='logloss'
)
xgb_model.fit(X_train_encoded, y_train)

# 6. ประเมินผล
y_pred_xgb = xgb_model.predict(X_test_encoded)

print("=== XGBoost Results ===")
print("Accuracy Score:", accuracy_score(y_test, y_pred_xgb))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb))

# 7. ดูความสำคัญของตัวแปรใน XGBoost
importances_xgb = pd.DataFrame({
    'Feature': X_train_encoded.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nTop 5 Most Important Features:\n", importances_xgb.head(5))

Note: you may need to restart the kernel to use updated packages.
=== XGBoost Results ===
Accuracy Score: 0.8181502488713972

Confusion Matrix:
 [[6492 1159]
 [ 412  576]]

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.85      0.89      7651
           1       0.33      0.58      0.42       988

    accuracy                           0.82      8639
   macro avg       0.64      0.72      0.66      8639
weighted avg       0.87      0.82      0.84      8639


Top 5 Most Important Features:
                Feature  Importance
38    poutcome_success    0.262508
25  contact_unrecorded    0.100910
32           month_mar    0.084782
35           month_oct    0.041320
36           month_sep    0.034899
